In [1]:
import numpy as np   
import pandas as pd

In [2]:
df=pd.read_csv('lalpurja_house_features_ready.csv')

In [3]:
df_lph=df.copy()

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

In [5]:
df_lph.sample(4)

,district,property_type,property_face,road_type,furnishing,municipality,ward_no,neighborhood,total_price,land_size_aana,...,log_built,floor_area_ratio,urban_centrality,amenity_access_score,house_size_score,comm_road_premium,neighborhood_x_district,municipality_x_ward,age_condition_score,ring_road_proximity
1235,1,1,7,0,1,Kageshwori Manohara,7.0,Kadaghari,32500000.0,3.10,...,7.696667,2.073564,0.058516,0.690093,10.859897,13.0,17.251648,120.220455,1.000000,0.120565
1577,2,1,0,1,2,Godawari,4.0,Godawari,19000000.0,6.25,...,6.935127,0.480000,0.053568,0.718663,13.738497,13.0,34.343398,68.628924,0.500000,0.111268
1017,1,0,0,0,2,Kathmandu,35.0,Koteshwor,37500000.0,4.00,...,6.935127,0.750000,0.077491,0.763013,11.161657,0.0,17.511171,616.726247,0.272727,0.188562
1259,1,1,4,0,2,Gokarneshwor,31.0,Mid Baneshowr,60000000.0,5.00,...,7.222566,0.800000,0.077072,0.810481,12.941101,13.0,17.702555,533.785357,0.130435,0.164850


In [6]:
# ─────────────────────────────────────────
# STEP 1 — DEFINE X AND Y
# Drop raw string columns — not useful for modeling
# Drop neighborhood and municipality raw strings
# since we have encoded versions already
# Target: log1p(total_price) — normalizes skewed prices
# All price values positive (min 10M) so log1p is safe
# ─────────────────────────────────────────
drop_for_model = ['total_price', 'neighborhood', 'municipality']
X = df_lph.drop(columns=drop_for_model)
y = np.log1p(df_lph['total_price'])

print(f"Features: {X.shape[1]}")
print(f"Samples:  {X.shape[0]}")
print(f"Target min: {y.min():.4f}, max: {y.max():.4f}")

Features: 34
Samples:  2190
Target min: 16.1181, max: 19.6734


In [7]:
# ─────────────────────────────────────────
# STEP 2 — TRAIN TEST SPLIT
# 80/20 — consistent with all previous models
# random_state=42 — reproducible results
# ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")


Training samples: 1752
Testing samples:  438


In [8]:
# ─────────────────────────────────────────
# STEP 3 — SCALE FOR LINEAR MODELS
# Tree models don't need scaling
# Linear models and SVR need same scale features
# ─────────────────────────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [9]:
# ─────────────────────────────────────────
# STEP 4 — TRAIN ALL MODELS
# ─────────────────────────────────────────
models = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=1.0),
    'Lasso':             Lasso(alpha=0.001),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost':           XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM':          lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
    'CatBoost':          CatBoostRegressor(n_estimators=100, random_state=42, verbose=0),
    'KNN':               KNeighborsRegressor(n_neighbors=5),
    'SVR':               SVR(kernel='rbf')
}

linear_models = ['Linear Regression', 'Ridge', 'Lasso', 'KNN', 'SVR']
results = []

for name, model in models.items():
    if name in linear_models:
        model.fit(X_train_sc, y_train)
        y_pred = model.predict(X_test_sc)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    results.append({
        'Model':      name,
        'RMSE (log)': round(rmse, 4),
        'MAE (log)':  round(mae, 4),
        'R² Score':   round(r2, 4),
        'Error %':    f"{(np.exp(mae) - 1) * 100:.1f}%"
    })
    print(f"✅ {name} done — R²: {r2:.4f}")

✅ Linear Regression done — R²: 0.5928
✅ Ridge done — R²: 0.5944
✅ Lasso done — R²: 0.5964
✅ Random Forest done — R²: 0.6072
✅ Gradient Boosting done — R²: 0.6263
✅ XGBoost done — R²: 0.6025
✅ LightGBM done — R²: 0.6206
✅ CatBoost done — R²: 0.6342
✅ KNN done — R²: 0.5712
✅ SVR done — R²: 0.6169


In [10]:
# ─────────────────────────────────────────
# STEP 5 — COMPARE RESULTS
# ─────────────────────────────────────────
results_df = pd.DataFrame(results).sort_values('R² Score', ascending=False)
print("\n=== Model Comparison ===")
print(results_df.to_string(index=False))


=== Model Comparison ===
            Model  RMSE (log)  MAE (log)  R² Score Error %
         CatBoost      0.3576     0.2223    0.6342   24.9%
Gradient Boosting      0.3615     0.2288    0.6263   25.7%
         LightGBM      0.3642     0.2319    0.6206   26.1%
              SVR      0.3660     0.2220    0.6169   24.9%
    Random Forest      0.3706     0.2311    0.6072   26.0%
          XGBoost      0.3728     0.2377    0.6025   26.8%
            Lasso      0.3757     0.2404    0.5964   27.2%
            Ridge      0.3766     0.2397    0.5944   27.1%
Linear Regression      0.3773     0.2402    0.5928   27.2%
              KNN      0.3872     0.2644    0.5712   30.3%


In [14]:
# Quick check — confirm neighborhood and municipality
# are dropped from X (we have encoded versions)
print("Columns in X:")
print([c for c in X.columns if 'neighborhood' in c or 'municipality' in c])

# Also check feature importance quickly
# to see if any features are hurting the model
feature_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': CatBoostRegressor(
        n_estimators=100, random_state=42, verbose=0
    ).fit(X_train, y_train).get_feature_importance()
}).sort_values('Importance', ascending=False)

print("\n=== Quick Feature Importance ===")
print(feature_imp.to_string(index=False))

Columns in X:
['neighborhood_encoded', 'municipality_encoded', 'neighborhood_x_district', 'municipality_x_ward']

=== Quick Feature Importance ===
                Feature  Importance
       house_size_score   19.025306
   neighborhood_encoded   14.403725
               log_land    9.910029
         land_size_aana    8.673944
              house_age    5.383200
        road_width_feet    3.365481
          property_type    2.993683
neighborhood_x_district    2.905694
             furnishing    2.618713
      comm_road_premium    2.564540
       floor_area_ratio    2.270072
   municipality_encoded    2.080794
    age_condition_score    1.943458
          bhatbhateni_m    1.923236
              log_built    1.687198
             hospital_m    1.655168
     public_transport_m    1.636446
       urban_centrality    1.490338
              college_m    1.386983
           boudhanath_m    1.320804
               school_m    1.320443
    municipality_x_ward    1.254191
              airport_m  

In [15]:
# ─────────────────────────────────────────
# DROP REDUNDANT AND WEAK FEATURES
# Why: Redundant features add noise without signal
# land_size_aana redundant with log_land
# built_up_sqft redundant with log_built and house_size_score
# is_corner_plot near zero importance
# ring_road_proximity redundant with ring_road_m and urban_centrality
# ─────────────────────────────────────────
drop_redundant = ['land_size_aana', 'built_up_sqft',
                  'is_corner_plot', 'ring_road_proximity']

X_clean = X.drop(columns=drop_redundant)
print(f"Features before: {X.shape[1]}")
print(f"Features after:  {X_clean.shape[1]}")
print(f"Dropped: {drop_redundant}")

# Retrain CatBoost quickly to verify improvement
from sklearn.model_selection import cross_val_score

cat_check = CatBoostRegressor(n_estimators=100, random_state=42, verbose=0)
scores = cross_val_score(cat_check, X_clean, y, cv=5, scoring='r2')
print(f"\n5-fold CV R² with cleaned features: {scores.mean():.4f} ± {scores.std():.4f}")

# Resplit with clean features
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clean, y, test_size=0.2, random_state=42
)
scaler_c     = StandardScaler()
X_train_sc_c = scaler_c.fit_transform(X_train_c)
X_test_sc_c  = scaler_c.transform(X_test_c)

# Quick comparison
results_clean = []
for name, model in models.items():
    if name in linear_models:
        model.fit(X_train_sc_c, y_train_c)
        y_pred = model.predict(X_test_sc_c)
    else:
        model.fit(X_train_c, y_train_c)
        y_pred = model.predict(X_test_c)

    r2 = r2_score(y_test_c, y_pred)
    mae = mean_absolute_error(y_test_c, y_pred)
    results_clean.append({
        'Model': name,
        'R² Score': round(r2, 4),
        'Error %': f"{(np.exp(mae) - 1) * 100:.1f}%"
    })
    print(f"✅ {name} — R²: {r2:.4f}")

results_clean_df = pd.DataFrame(results_clean).sort_values('R² Score', ascending=False)
print("\n=== Cleaned Features Comparison ===")
print(results_clean_df.to_string(index=False))

Features before: 34
Features after:  30
Dropped: ['land_size_aana', 'built_up_sqft', 'is_corner_plot', 'ring_road_proximity']

5-fold CV R² with cleaned features: 0.6151 ± 0.0458
✅ Linear Regression — R²: 0.5940
✅ Ridge — R²: 0.5978
✅ Lasso — R²: 0.6038
✅ Random Forest — R²: 0.6066
✅ Gradient Boosting — R²: 0.6291
✅ XGBoost — R²: 0.6323
✅ LightGBM — R²: 0.6279
✅ CatBoost — R²: 0.6563
✅ KNN — R²: 0.5789
✅ SVR — R²: 0.6216

=== Cleaned Features Comparison ===
            Model  R² Score Error %
         CatBoost    0.6563   24.5%
          XGBoost    0.6323   25.9%
Gradient Boosting    0.6291   25.6%
         LightGBM    0.6279   26.1%
              SVR    0.6216   24.8%
    Random Forest    0.6066   25.9%
            Lasso    0.6038   26.9%
            Ridge    0.5978   27.2%
Linear Regression    0.5940   27.3%
              KNN    0.5789   30.4%


In [16]:
from sklearn.model_selection import RandomizedSearchCV

param_grids = {
    'CatBoost': {
        'model': CatBoostRegressor(random_state=42, verbose=0),
        'params': {
            'iterations':    [500, 1000, 1500, 2000],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'depth':         [4, 6, 8],
            'l2_leaf_reg':   [1, 3, 5, 9]
        }
    },
    'XGBoost': {
        'model': XGBRegressor(random_state=42, verbosity=0),
        'params': {
            'n_estimators':     [200, 300, 500],
            'learning_rate':    [0.01, 0.05, 0.1],
            'max_depth':        [3, 5, 7],
            'subsample':        [0.7, 0.8, 1.0],
            'colsample_bytree': [0.7, 0.8, 1.0],
            'gamma':            [0, 0.1, 0.2]
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {
            'n_estimators':  [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [3, 5, 7],
            'subsample':     [0.7, 0.8, 1.0]
        }
    }
}

tuned_models  = {}
tuned_results = []

for name, config in param_grids.items():
    print(f"🔍 Tuning {name}...")

    search = RandomizedSearchCV(
        estimator=config['model'],
        param_distributions=config['params'],
        n_iter=50, cv=5, scoring='r2',
        random_state=42, n_jobs=-1
    )

    search.fit(X_train_c, y_train_c)
    tuned_models[name] = search.best_estimator_
    y_pred = tuned_models[name].predict(X_test_c)

    rmse = np.sqrt(mean_squared_error(y_test_c, y_pred))
    mae  = mean_absolute_error(y_test_c, y_pred)
    r2   = r2_score(y_test_c, y_pred)

    tuned_results.append({
        'Model':       name,
        'RMSE (log)':  round(rmse, 4),
        'MAE (log)':   round(mae, 4),
        'R² Score':    round(r2, 4),
        'Error %':     f"{(np.exp(mae) - 1) * 100:.1f}%",
        'Best Params': search.best_params_
    })
    print(f"✅ {name} done — R²: {r2:.4f}")

tuned_df = pd.DataFrame(tuned_results).sort_values('R² Score', ascending=False)
print("\n=== Tuned Model Comparison ===")
print(tuned_df[['Model','RMSE (log)','MAE (log)','R² Score','Error %']].to_string(index=False))

print("\n=== Best Parameters ===")
for result in tuned_results:
    print(f"\n{result['Model']}:")
    for param, value in result['Best Params'].items():
        print(f"  {param}: {value}")

🔍 Tuning CatBoost...
✅ CatBoost done — R²: 0.6475
🔍 Tuning XGBoost...
✅ XGBoost done — R²: 0.6470
🔍 Tuning Gradient Boosting...
✅ Gradient Boosting done — R²: 0.6344

=== Tuned Model Comparison ===
            Model  RMSE (log)  MAE (log)  R² Score Error %
         CatBoost      0.3511     0.2174    0.6475   24.3%
          XGBoost      0.3513     0.2228    0.6470   25.0%
Gradient Boosting      0.3575     0.2222    0.6344   24.9%

=== Best Parameters ===

CatBoost:
  learning_rate: 0.03
  l2_leaf_reg: 9
  iterations: 1500
  depth: 6

XGBoost:
  subsample: 0.8
  n_estimators: 500
  max_depth: 3
  learning_rate: 0.1
  gamma: 0.2
  colsample_bytree: 1.0

Gradient Boosting:
  subsample: 0.7
  n_estimators: 200
  max_depth: 5
  learning_rate: 0.05


In [17]:
# ─────────────────────────────────────────
# RETRAIN BEST MODEL WITH PRE-TUNED PARAMS
# Pre-tuned CatBoost R² 0.656 > tuned 0.648
# Using n_estimators=100 was already good
# But let's try with more iterations since
# lalpurja land benefited from 1500 iterations
# ─────────────────────────────────────────
best_model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=9,
    random_state=42,
    verbose=0
)
best_model.fit(X_train_c, y_train_c)
y_pred_best = best_model.predict(X_test_c)

r2   = r2_score(y_test_c, y_pred_best)
mae  = mean_absolute_error(y_test_c, y_pred_best)
rmse = np.sqrt(mean_squared_error(y_test_c, y_pred_best))

print(f"=== Final Lalpurja Housing Model ===")
print(f"R² Score:  {r2:.4f}")
print(f"MAE (log): {mae:.4f}")
print(f"RMSE(log): {rmse:.4f}")
print(f"Error %:   {(np.exp(mae)-1)*100:.1f}%")

=== Final Lalpurja Housing Model ===
R² Score:  0.6475
MAE (log): 0.2174
RMSE(log): 0.3511
Error %:   24.3%


In [18]:
# ─────────────────────────────────────────
# FEATURE IMPORTANCE
# ─────────────────────────────────────────
feature_imp = pd.DataFrame({
    'Feature':    X_clean.columns,
    'Importance': best_model.get_feature_importance()
}).sort_values('Importance', ascending=False)

print(f"\n=== Feature Importance ===")
print(feature_imp.to_string(index=False))


=== Feature Importance ===
                Feature  Importance
               log_land   22.764894
       house_size_score   13.892297
   neighborhood_encoded   13.779423
              house_age    4.403381
neighborhood_x_district    4.156395
        road_width_feet    3.833405
       floor_area_ratio    3.686669
             furnishing    3.349906
          property_type    3.044131
    age_condition_score    2.282866
              log_built    2.276440
      comm_road_premium    2.030789
          bhatbhateni_m    1.667827
   municipality_encoded    1.646096
            ring_road_m    1.614927
     public_transport_m    1.564280
           boudhanath_m    1.551474
       urban_centrality    1.544558
    municipality_x_ward    1.381873
             hospital_m    1.145463
   amenity_access_score    1.130385
             pharmacy_m    1.032245
              college_m    0.990620
       police_station_m    0.936519
              airport_m    0.896046
               school_m    0.874798


In [19]:
# ─────────────────────────────────────────
# SAVE MODEL
# ─────────────────────────────────────────
import joblib
joblib.dump(best_model, 'catboost_lalpurja_house_model_final.pkl')

# Verify
loaded = joblib.load('catboost_lalpurja_house_model_final.pkl')
sample = np.expm1(loaded.predict(X_test_c[:3]))
print(f"\n✅ Model saved: catboost_lalpurja_house_model_final.pkl")
print(f"✅ Sample predictions (NPR): {sample.astype(int)}")

print(f"\n=== Lalpurja Housing Model Summary ===")
print(f"Algorithm:     CatBoost")
print(f"R² Score:      {r2:.4f}")
print(f"Average Error: {(np.exp(mae)-1)*100:.1f}%")
print(f"Training rows: {X_train_c.shape[0]}")
print(f"Testing rows:  {X_test_c.shape[0]}")
print(f"Features:      {X_clean.shape[1]}")


✅ Model saved: catboost_lalpurja_house_model_final.pkl
✅ Sample predictions (NPR): [37008676 43409840 32480258]

=== Lalpurja Housing Model Summary ===
Algorithm:     CatBoost
R² Score:      0.6475
Average Error: 24.3%
Training rows: 1752
Testing rows:  438
Features:      30
